# σ-Loss Analysis — DiT-mini Parity Rep2

DSM loss decomposed by noise level σ across training checkpoints for the parity rep2 series.

**Three data splits:**
- *Train* (blue): in-distribution training samples (N=4096)
- *Test* (red): freshly sampled parity-valid samples, non-overlapping with train
- *Random* (gray): uniform ±1 boolean hypercube, no rule

**Interpretation:**
- Low σ (fine scale): tests whether the model has memorized precise data coordinates
- Mid σ (σ ≈ 0.5–2): tests whether the model has learned the parity rule structure
- High σ (σ > 20): all splits converge (uninformative noise regime)
- Train/test gap at mid σ → memorization; train ≈ test → generalization

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
SAVEROOT  = "/n/holylfs06/LABS/kempner_fellow_binxuwang/Users/binxuwang/DL_Projects/DiffusionParityLearning"
FIGDIR    = "../figures/sigma_loss_parity_rep2"
SAVE_FIGS = True

G_VALUES  = [2, 3, 4, 6, 9, 12, 18, 36]
EXP_TMPL  = "DiT_mini_parity_N4096_D36_G{g}_even_rep2"

SIGMA_BINS = [
    ("σ ∈ [0.002, 0.02]",  0.002,  0.02),
    ("σ ∈ [0.02, 0.2]",    0.02,   0.2),
    ("σ ∈ [0.2, 2]",       0.2,    2.0),
    ("σ ∈ [2, 20]",        2.0,   20.0),
    ("σ ∈ [20, 80]",      20.0,   80.0),
]

SPLIT_STYLES = {
    "train":  dict(color="#2166ac", lw=1.8, label="Train (in-dist)"),
    "test":   dict(color="#d73027", lw=1.8, label="Test (valid, unseen)"),
    "random": dict(color="#555555", lw=1.2, ls="--", label="Random ±1"),
}

os.makedirs(FIGDIR, exist_ok=True)

## 1. Load All σ-Loss Data

In [ ]:
def load_sigma_data(exp_name):
    sigma_dir = os.path.join(SAVEROOT, exp_name, "sigma_loss")
    records = []
    for fname in sorted(os.listdir(sigma_dir)):
        if not fname.endswith(".npz"):
            continue
        d = np.load(os.path.join(sigma_dir, fname), allow_pickle=True)
        if len(d["sigma_grid"]) < 10:
            continue
        records.append({
            "epoch":        int(d["meta_ckpt_epoch"]),
            "sigma_grid":   d["sigma_grid"],
            "loss_train":   d["loss_train"],
            "loss_test":    d["loss_test"],
            "loss_random":  d["loss_random"],
            "std_train":    d["std_train"],
            "std_test":     d["std_test"],
            "std_random":   d["std_random"],
        })
    records.sort(key=lambda r: r["epoch"])
    return records


def bin_mean(loss, sigma_grid, smin, smax):
    mask = (sigma_grid >= smin) & (sigma_grid <= smax)
    return float(loss[mask].mean()) if mask.any() else np.nan


all_data = {}
for g in G_VALUES:
    exp = EXP_TMPL.format(g=g)
    try:
        all_data[g] = load_sigma_data(exp)
        print(f"G={g:2d}: {len(all_data[g])} checkpoints, "
              f"epochs {all_data[g][0]['epoch']} → {all_data[g][-1]['epoch']}")
    except Exception as e:
        print(f"G={g:2d}: FAILED — {e}")
        all_data[g] = []

## 2. σ-bin Evolution: Loss vs Training Step per Bin

In [ ]:
def plot_evolution(g, log_loss=False, save=False):
    records = all_data[g]
    if not records:
        print(f"G={g}: no data"); return

    epochs = np.array([r["epoch"] for r in records])
    n_bins = len(SIGMA_BINS)

    fig, axes = plt.subplots(n_bins + 1, 1, figsize=(12, 2.8 * (n_bins + 1)), sharex=True)
    fig.suptitle(f"G={g} σ-bin Loss Evolution (rep2, N=4096)", fontsize=13, fontweight="bold")

    # Top panel: overall loss across all σ at final checkpoint
    ax0 = axes[0]
    final = records[-1]
    sg = final["sigma_grid"]
    for split in ["train", "test", "random"]:
        kw = SPLIT_STYLES[split].copy(); lbl = kw.pop("label")
        ax0.plot(sg, final[f"loss_{split}"], **kw, label=lbl)
    ax0.set_xscale("log"); ax0.set_yscale("log")
    ax0.set_ylabel("Loss (final ckpt)")
    ax0.set_title(f"Final σ-curve (epoch {final['epoch']})")
    ax0.legend(fontsize=9); ax0.grid(True, which="both", ls="--", alpha=0.4)

    # Per-bin panels
    for b_idx, (bin_label, smin, smax) in enumerate(SIGMA_BINS):
        ax = axes[b_idx + 1]
        for split in ["train", "test", "random"]:
            kw = SPLIT_STYLES[split].copy(); lbl = kw.pop("label")
            vals = np.array([bin_mean(r[f"loss_{split}"], r["sigma_grid"], smin, smax)
                             for r in records])
            ax.plot(epochs, vals, **kw, label=lbl)
        ax.set_xscale("log")
        if log_loss: ax.set_yscale("log")
        ax.set_ylabel("Mean loss")
        ax.set_title(bin_label)
        ax.legend(fontsize=8); ax.grid(True, which="both", ls="--", alpha=0.4)

    axes[-1].set_xlabel("Training step")
    plt.tight_layout()
    if save:
        path = os.path.join(FIGDIR, f"evolution_G{g:02d}{'_logy' if log_loss else ''}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved {path}")
    plt.show()


# Plot G=2 and G=9 as representative cases
for g in [2, 9]:
    plot_evolution(g, log_loss=False, save=SAVE_FIGS)
    plot_evolution(g, log_loss=True,  save=SAVE_FIGS)

## 3. All G — Evolution of σ ∈ [0.2, 2] Bin (Rule-Learning Region)

In [ ]:
SMIN, SMAX = 0.2, 2.0   # the mid-σ bin most sensitive to rule learning

cmap   = plt.cm.viridis
colors = {g: cmap(i / (len(G_VALUES) - 1)) for i, g in enumerate(G_VALUES)}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"All G — σ ∈ [{SMIN}, {SMAX}] Loss vs Training Step", fontsize=13, fontweight="bold")

for g in G_VALUES:
    if not all_data[g]: continue
    records = all_data[g]
    epochs  = np.array([r["epoch"] for r in records])
    c       = colors[g]
    label   = f"G={g}"
    for ax_i, split in enumerate(["train", "test", "random"]):
        vals = np.array([bin_mean(r[f"loss_{split}"], r["sigma_grid"], SMIN, SMAX)
                         for r in records])
        axes[ax_i].plot(epochs, vals, color=c, lw=1.4, label=label)

titles = ["Train loss", "Test loss (parity-valid, unseen)", "Random loss (boolean cube)"]
for ax, title in zip(axes, titles):
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Mean loss in σ ∈ [0.2, 2]")
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGDIR, "all_G_mid_sigma_overlay.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Train / Test Gap Evolution — Memorization Signal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Train/Test Loss Gap — Memorization Signal", fontsize=13, fontweight="bold")

for g in G_VALUES:
    if not all_data[g]: continue
    records = all_data[g]
    epochs  = np.array([r["epoch"] for r in records])
    c       = colors[g]

    # Gap in mid-σ bin (rule-learning region)
    gap_mid = np.array([
        bin_mean(r["loss_test"], r["sigma_grid"], 0.2, 2.0) -
        bin_mean(r["loss_train"], r["sigma_grid"], 0.2, 2.0)
        for r in records
    ])
    # Gap in fine-σ bin (memorization of exact coordinates)
    gap_fine = np.array([
        bin_mean(r["loss_test"], r["sigma_grid"], 0.002, 0.02) -
        bin_mean(r["loss_train"], r["sigma_grid"], 0.002, 0.02)
        for r in records
    ])

    axes[0].plot(epochs, gap_mid,  color=c, lw=1.4, label=f"G={g}")
    axes[1].plot(epochs, gap_fine, color=c, lw=1.4, label=f"G={g}")

for ax, title, ylabel in [
    (axes[0], "Mid-σ gap [0.2, 2] — Rule generalization",   "Test − Train loss"),
    (axes[1], "Fine-σ gap [0.002, 0.02] — Exact memorization", "Test − Train loss"),
]:
    ax.set_xscale("log")
    ax.set_xlabel("Training step")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.axhline(0, color="k", lw=0.8, ls=":")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, which="both", ls="--", alpha=0.4)

plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGDIR, "train_test_gap_evolution.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Time-Heatmap: σ × training-step, colored by loss

In [ ]:
from scipy.interpolate import RegularGridInterpolator

def make_sigma_step_heatmap(g, split="train"):
    records = all_data[g]
    if not records: return None, None, None
    epochs = np.array([r["epoch"] for r in records])
    sigma  = records[0]["sigma_grid"]
    mat    = np.array([r[f"loss_{split}"] for r in records])  # (T, S)
    return epochs, sigma, mat


# Show G=2 and G=36 side-by-side, all 3 splits
show_Gs = [2, 9, 36]
splits  = ["train", "test", "random"]

fig, axes = plt.subplots(len(show_Gs), len(splits),
                          figsize=(5 * len(splits), 4 * len(show_Gs)))
fig.suptitle("σ-Loss Heatmap: x=σ, y=training step, color=loss", fontsize=13)

for row, g in enumerate(show_Gs):
    epochs, sigma, mat_train = make_sigma_step_heatmap(g, "train")
    if epochs is None: continue
    _, _, mat_test   = make_sigma_step_heatmap(g, "test")
    _, _, mat_random = make_sigma_step_heatmap(g, "random")
    mats = [mat_train, mat_test, mat_random]

    vmax = float(np.nanpercentile(mat_test, 95))
    vmax = min(vmax, 1.5)

    for col, (split, mat) in enumerate(zip(splits, mats)):
        ax = axes[row, col]
        # pcolormesh: x=sigma, y=epoch
        im = ax.pcolormesh(
            sigma, np.arange(len(epochs) + 1), mat,
            cmap="RdYlGn_r", vmin=0, vmax=vmax, shading="flat"
        )
        ax.set_xscale("log")
        ax.set_xlabel("σ")
        ax.set_ylabel("Checkpoint index")
        ax.set_title(f"G={g} — {split}")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="loss")
        # Mark epoch ticks at round numbers
        tick_idx = [i for i, e in enumerate(epochs) if e in
                    [1000, 10000, 100000, 1000000] or i in [0, len(epochs)//2, len(epochs)-1]]
        ax.set_yticks([i + 0.5 for i in tick_idx])
        ax.set_yticklabels([f"{epochs[i]:.0e}" for i in tick_idx], fontsize=7)

plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGDIR, "sigma_step_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 6. Final σ-Curves: All G in One Figure

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Final checkpoint σ-Loss Curves — All G (rep2)", fontsize=13, fontweight="bold")

for ax, g in zip(axes.flat, G_VALUES):
    records = all_data[g]
    if not records: ax.set_visible(False); continue
    final = records[-1]
    sg    = final["sigma_grid"]

    for split in ["train", "test", "random"]:
        kw = SPLIT_STYLES[split].copy(); lbl = kw.pop("label")
        ax.plot(sg, final[f"loss_{split}"], **kw, label=lbl)
        lo = np.maximum(final[f"loss_{split}"] - final[f"std_{split}"], 1e-8)
        hi = final[f"loss_{split}"] + final[f"std_{split}"]
        ax.fill_between(sg, lo, hi, alpha=0.15, color=kw["color"])

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("σ"); ax.set_ylabel("Loss")
    ax.set_title(f"G={g}  (epoch {final['epoch']})")
    ax.legend(fontsize=7)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    # Shade mid-sigma region
    ax.axvspan(0.2, 2.0, alpha=0.08, color="gold", label="mid-σ")

plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGDIR, "sigma_curves_final_all_G.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Multi-Checkpoint σ-Curves for Representative G Values

In [ ]:
def plot_multicheckpoint(g, which_idx=None, save=False):
    records = all_data[g]
    if not records: return
    if which_idx is None:
        # Pick ~5 evenly spaced checkpoints
        n = len(records)
        which_idx = sorted(set([0, n//4, n//2, 3*n//4, n-1]))

    n_ckpt = len(which_idx)
    fig, axes = plt.subplots(1, n_ckpt, figsize=(4.5 * n_ckpt, 4), sharey=True)
    fig.suptitle(f"G={g} — σ-Curves at Multiple Checkpoints", fontsize=12, fontweight="bold")
    if n_ckpt == 1: axes = [axes]

    for ax, idx in zip(axes, which_idx):
        rec = records[idx]
        sg  = rec["sigma_grid"]
        for split in ["train", "test", "random"]:
            kw = SPLIT_STYLES[split].copy(); lbl = kw.pop("label")
            ax.plot(sg, rec[f"loss_{split}"], **kw, label=lbl)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("σ")
        ax.set_title(f"step {rec['epoch']:,}")
        ax.legend(fontsize=8)
        ax.grid(True, which="both", ls="--", alpha=0.4)
        ax.axvspan(0.2, 2.0, alpha=0.08, color="gold")

    axes[0].set_ylabel("Loss")
    plt.tight_layout()
    if save:
        path = os.path.join(FIGDIR, f"sigma_multickpt_G{g:02d}.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved {path}")
    plt.show()


for g in [2, 6, 36]:
    plot_multicheckpoint(g, save=SAVE_FIGS)

## 8. All G — σ-Heatmap: x=log-step, y=G, color=gap (test−train)

In [ ]:
from scipy.interpolate import interp1d

log_steps_grid = np.logspace(0, 6, 400)

def make_gap_row(g, smin, smax):
    records = all_data[g]
    if not records: return np.full(len(log_steps_grid), np.nan)
    epochs = np.array([r["epoch"] for r in records], dtype=float)
    gaps   = np.array([
        bin_mean(r["loss_test"], r["sigma_grid"], smin, smax) -
        bin_mean(r["loss_train"], r["sigma_grid"], smin, smax)
        for r in records
    ])
    f = interp1d(epochs, gaps, bounds_error=False, fill_value=(gaps[0], gaps[-1]))
    return f(log_steps_grid)


fig, axes = plt.subplots(1, len(SIGMA_BINS), figsize=(5 * len(SIGMA_BINS), 4))
fig.suptitle("Test−Train Gap Heatmap: x=log-step, y=G, color=gap", fontsize=13)

for ax, (bin_label, smin, smax) in zip(axes, SIGMA_BINS):
    mat = np.array([make_gap_row(g, smin, smax) for g in G_VALUES])
    vabs = np.nanpercentile(np.abs(mat), 95)
    im = ax.pcolormesh(
        log_steps_grid, np.arange(len(G_VALUES) + 1), mat,
        cmap="RdBu_r", vmin=-vabs, vmax=vabs, shading="flat"
    )
    ax.set_xscale("log")
    ax.set_xlabel("Training step")
    ax.set_yticks(np.arange(len(G_VALUES)) + 0.5)
    ax.set_yticklabels([f"G={g}" for g in G_VALUES])
    ax.set_title(bin_label, fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="test−train")

plt.tight_layout()
if SAVE_FIGS:
    fig.savefig(os.path.join(FIGDIR, "gap_heatmap_all_G.png"), dpi=150, bbox_inches="tight")
plt.show()